# Работа с данными

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)


In [2]:
path = 'data_raw/ETHUSDT_futures_um_1d_with_funding_2020-01_2025-12.csv'

df = pd.read_csv(path)
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], utc=True, errors='coerce')
df = df.sort_values('datetime_utc').reset_index(drop=True)
df.shape

(2192, 13)

In [3]:
df.head(3)

,datetime_utc,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote,date_utc,funding_rate_day_sum,funding_points_in_day
0,2020-01-01 00:00:00+00:00,129.12,132.96,128.62,130.62,466063.929,6.103756e+07,40168,232492.871,3.044614e+07,2020-01-01,0.000012,3
1,2020-01-02 00:00:00+00:00,130.63,130.75,126.25,127.11,523582.924,6.737797e+07,47479,253840.635,3.267493e+07,2020-01-02,0.000250,3
2,2020-01-03 00:00:00+00:00,127.11,135.07,125.78,134.29,951123.597,1.250189e+08,106861,486159.153,6.390715e+07,2020-01-03,0.000300,3


In [4]:
df.tail(3)

,datetime_utc,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote,date_utc,funding_rate_day_sum,funding_points_in_day
2189,2025-12-29 00:00:00+00:00,2949.60,3056.00,2908.58,2936.45,4975460.686,1.477776e+10,7102902,2503963.457,7.437437e+09,2025-12-29,0.000195,3
2190,2025-12-30 00:00:00+00:00,2936.45,3008.67,2917.48,2972.22,3160478.075,9.386429e+09,4897272,1569781.659,4.662351e+09,2025-12-30,0.000124,3
2191,2025-12-31 00:00:00+00:00,2972.23,3027.73,2957.59,2970.40,2411514.797,7.193093e+09,3583480,1225250.687,3.654995e+09,2025-12-31,0.000099,3


In [5]:
checks = {}

checks['rows'] = len(df)
checks['datetime_nulls'] = int(df['datetime_utc'].isna().sum())
checks['datetime_duplicates'] = int(df['datetime_utc'].duplicated().sum())
checks['is_sorted'] = bool(df['datetime_utc'].is_monotonic_increasing)

dt = df['datetime_utc'].dropna()
if len(dt) > 1:
    gaps = dt.diff().dt.total_seconds().div(86400)
    gap_counts = gaps.value_counts(dropna=True).sort_index()
    checks['min_gap_days'] = float(gaps.min())
    checks['max_gap_days'] = float(gaps.max())
else:
    gap_counts = pd.Series(dtype='int64')

checks, gap_counts.head(10), gap_counts.tail(10)


({'rows': 2192,
  'datetime_nulls': 0,
  'datetime_duplicates': 0,
  'is_sorted': True,
  'min_gap_days': 1.0,
  'max_gap_days': 1.0},
 datetime_utc
 1.0    2191
 Name: count, dtype: int64,
 datetime_utc
 1.0    2191
 Name: count, dtype: int64)

In [6]:
num_cols = [
    'open', 'high', 'low', 'close', 'volume',
    'quote_volume', 'taker_buy_base', 'taker_buy_quote', 'funding_rate_day_sum'
]
int_cols = ['trades']

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['trades'] = pd.to_numeric(df['trades'], errors='coerce').astype('Int64')

before = len(df)
df = df.dropna(subset=['datetime_utc', 'open', 'high', 'low', 'close']).copy()
after = len(df)

bad_high = (df['high'] < df[['open', 'close']].max(axis=1)).sum()
bad_low = (df['low'] > df[['open', 'close']].min(axis=1)).sum()
nonpos_close = (df['close'] <= 0).sum()

{
    'dropped_rows_due_to_nans': before - after,
    'bad_high_count': int(bad_high),
    'bad_low_count': int(bad_low),
    'nonpositive_close_count': int(nonpos_close)
}


{'dropped_rows_due_to_nans': 0,
 'bad_high_count': 0,
 'bad_low_count': 0,
 'nonpositive_close_count': 0}

Доходность, лог-доходность, дневной ценовой размах, True Range, Average True Range, Годовая волатильность на основе доходностей

In [7]:
df = df.set_index('datetime_utc').sort_index()

df['ret_1d'] = df['close'].pct_change()
df['logret_1d'] = np.log(df['close']).diff()

df['range'] = (df['high'] - df['low']) / df['close'].shift(1)

prev_close = df['close'].shift(1)
tr = pd.concat(
    [
        (df['high'] - df['low']),
        (df['high'] - prev_close).abs(),
        (df['low'] - prev_close).abs()
    ],
    axis=1
).max(axis=1)

df['true_range'] = tr
df['atr_14'] = df['true_range'].rolling(14, min_periods=14).mean()

df['vol_30'] = df['ret_1d'].rolling(30, min_periods=30).std() * np.sqrt(365)

df[['open', 'high', 'low', 'close', 'volume', 'ret_1d', 'logret_1d', 'atr_14', 'vol_30']].tail(10)


,open,high,low,close,volume,ret_1d,logret_1d,atr_14,vol_30
datetime_utc,,,,,,,,,
2025-12-22 00:00:00+00:00,3000.61,3077.99,2961.00,3008.49,4028005.767,0.002623,0.002619,166.452143,0.632509
2025-12-23 00:00:00+00:00,3008.50,3035.00,2899.30,2963.29,4025936.154,-0.015024,-0.015138,154.110000,0.634951
2025-12-24 00:00:00+00:00,2963.29,2976.48,2886.75,2946.23,2622651.930,-0.005757,-0.005774,149.123571,0.607372
2025-12-25 00:00:00+00:00,2946.23,2969.99,2888.88,2902.32,1833895.842,-0.014904,-0.015016,141.787143,0.609673
2025-12-26 00:00:00+00:00,2902.32,2994.00,2892.72,2926.82,3511943.959,0.008442,0.008406,133.044286,0.605019
2025-12-27 00:00:00+00:00,2926.83,2959.84,2914.63,2947.61,899512.695,0.007103,0.007078,131.987857,0.605499
2025-12-28 00:00:00+00:00,2947.61,2959.99,2922.84,2949.60,1156340.657,0.000675,0.000675,127.060000,0.605179
2025-12-29 00:00:00+00:00,2949.60,3056.00,2908.58,2936.45,4975460.686,-0.004458,-0.004468,117.126429,0.603450
2025-12-30 00:00:00+00:00,2936.45,3008.67,2917.48,2972.22,3160478.075,0.012181,0.012108,115.632857,0.604969


In [8]:
df[['open', 'high', 'low', 'close', 'volume', 'ret_1d', 'logret_1d', 'atr_14', 'vol_30']].head(31)

,open,high,low,close,volume,ret_1d,logret_1d,atr_14,vol_30
datetime_utc,,,,,,,,,
2020-01-01 00:00:00+00:00,129.12,132.96,128.62,130.62,466063.929,NaN,NaN,NaN,NaN
2020-01-02 00:00:00+00:00,130.63,130.75,126.25,127.11,523582.924,-0.026872,-0.027239,NaN,NaN
2020-01-03 00:00:00+00:00,127.11,135.07,125.78,134.29,951123.597,0.056487,0.054949,NaN,NaN
2020-01-04 00:00:00+00:00,134.31,135.80,132.41,134.14,509778.108,-0.001117,-0.001118,NaN,NaN
2020-01-05 00:00:00+00:00,134.14,138.17,134.12,135.33,669677.687,0.008871,0.008832,NaN,NaN
2020-01-06 00:00:00+00:00,135.32,144.38,134.82,144.04,1095147.166,0.064361,0.062375,NaN,NaN
2020-01-07 00:00:00+00:00,144.11,145.25,138.70,142.86,963331.505,-0.008192,-0.008226,NaN,NaN
2020-01-08 00:00:00+00:00,142.86,147.94,137.15,140.75,1208133.766,-0.014770,-0.014880,NaN,NaN
2020-01-09 00:00:00+00:00,140.76,141.52,135.33,137.73,956864.213,-0.021456,-0.021690,NaN,NaN


In [9]:
keep_cols = [
    'open', 'high', 'low', 'close', 'volume',
    'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote',
    'ret_1d', 'logret_1d', 'range', 'true_range', 'atr_14', 'vol_30', 'funding_rate_day_sum'
]

df_bt = df[keep_cols].copy()

required = ['ret_1d', 'atr_14', 'vol_30']
df_bt_clean = df_bt.dropna(subset=required).copy()

df_bt_clean.shape

(2162, 16)

In [10]:
out_path = 'data/eth_futures_2020-2025.csv'
df_bt_clean.to_csv(out_path, index=True)
out_path

'data/eth_futures_2020-2025.csv'

In [13]:
df_bt_clean

,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote,ret_1d,logret_1d,range,true_range,atr_14,vol_30,funding_rate_day_sum
datetime_utc,,,,,,,,,,,,,,,,
2020-01-31 00:00:00+00:00,185.16,186.34,175.50,180.14,672522.049,1.217445e+08,82806,314492.275,5.692520e+07,-0.027007,-0.027378,0.058550,10.84,9.131429,0.767391,0.003083
2020-02-01 00:00:00+00:00,180.14,184.58,179.41,183.82,526688.019,9.608089e+07,73274,249313.968,4.550130e+07,0.020429,0.020223,0.028700,5.17,8.465714,0.755251,0.001956
2020-02-02 00:00:00+00:00,183.80,193.70,179.00,188.59,907342.798,1.708467e+08,136351,466111.723,8.779435e+07,0.025949,0.025618,0.079970,14.70,8.329286,0.740535,0.000613
2020-02-03 00:00:00+00:00,188.64,195.45,186.65,189.92,652054.538,1.241221e+08,91435,316307.928,6.025571e+07,0.007052,0.007028,0.046662,8.80,8.387857,0.739251,0.000300
2020-02-04 00:00:00+00:00,189.89,191.76,184.71,189.01,614290.379,1.153646e+08,80668,291973.564,5.483796e+07,-0.004791,-0.004803,0.037121,7.05,8.511429,0.741592,0.000907
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27 00:00:00+00:00,2926.83,2959.84,2914.63,2947.61,899512.695,2.636508e+09,1352404,447307.621,1.311245e+09,0.007103,0.007078,0.015447,45.21,131.987857,0.605499,0.000117
2025-12-28 00:00:00+00:00,2947.61,2959.99,2922.84,2949.60,1156340.657,3.400544e+09,1882393,563046.755,1.656079e+09,0.000675,0.000675,0.012603,37.15,127.060000,0.605179,0.000179
2025-12-29 00:00:00+00:00,2949.60,3056.00,2908.58,2936.45,4975460.686,1.477776e+10,7102902,2503963.457,7.437437e+09,-0.004458,-0.004468,0.049980,147.42,117.126429,0.603450,0.000195
